> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 10 · Notebook 03 — Triple-barrier labels, uniqueness and meta-labels

**Sessions:** S5–S6 (Triple-barrier labels & sample weights · Meta-labeling & bet sizing) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Label events with profit-taking, stop-loss and time barriers scaled by volatility.
2. Measure how much labels overlap, and weight them by uniqueness.
3. Turn a primary rule into meta-labels: "should I take this trade?"
4. Size bets from a probability.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. Three barriers

At each event `t0`, follow the price over the next `max_hold` bars: `path = close / close[t0] − 1`. The upper barrier is `pt × vol[t0]`, the lower `−sl × vol[t0]` (volatility known at `t0`). The label ends at `t1`, the **first** bar that touches either barrier, or the last bar of the path if neither is touched. Label `+1` upper, `−1` lower, `0` time. `hit_up` and `hit_dn` are the first touching dates (NaT if none): take the earliest one that exists.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def triple_barrier(close, events, vol, pt=1.0, sl=1.0, max_hold=10):
    out, idx = [], close.index
    for t0 in events:
        i0 = idx.get_loc(t0)
        if i0 + 1 >= len(idx) or np.isnan(vol.loc[t0]):
            continue
        path = close.iloc[i0 + 1: i0 + 1 + max_hold] / close.iloc[i0] - 1
        up, dn = pt * vol.loc[t0], -sl * vol.loc[t0]
        hit_up, hit_dn = path[path >= up].index.min(), path[path <= dn].index.min()
        t1 = min([t for t in (hit_up, hit_dn) if pd.notna(t)], default=path.index[-1])
        label = 1 if t1 == hit_up else -1 if t1 == hit_dn else 0
        out.append((t0, t1, path.loc[t1], label))
    return pd.DataFrame(out, columns=["t0", "t1", "ret", "label"]).set_index("t0")

mk = p.regime_market()
close = mk["close"]
events = p.cusum_events(close, 0.015)
events = events[events >= close.index[100]]
vol = p.daily_vol(close)
cases = [(close, events, vol), (close, events[:200], vol, 2.0, 1.0, 5)]
mine = [p.attempt(triple_barrier, *c) for c in cases]
mine = p.check("triple_barrier", mine, [p.triple_barrier(*c) for c in cases])
tb = mine[0]
print(f"{len(tb)} labels: {tb['label'].value_counts().to_dict()}  (+1 upper, −1 lower, 0 time)")

In [ ]:
t0 = tb.index[40]
i0 = close.index.get_loc(t0)
seg = close.iloc[i0: i0 + 12] / close.loc[t0] - 1
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(seg.index, seg, marker="o", lw=1)
ax.axhline(vol.loc[t0], color="#1baf7a", ls="--", label="profit-taking"); ax.axhline(-vol.loc[t0], color="#e34948", ls="--", label="stop-loss")
ax.axvline(close.index[i0 + 10], color="#8a8984", ls=":", label="time barrier"); ax.axvline(tb.loc[t0, "t1"], color="black", lw=1, label="t1")
ax.legend(fontsize=8); ax.set_title(f"one event: label {tb.loc[t0, 'label']:+d}"); plt.show()

## 2. Overlapping labels are not independent

Two labels that span the same days share information. For each bar, count how many labels are active (`p.num_concurrent`). A label's **average uniqueness** is the mean of `1 / concurrency` over its own bars `[t0, t1]`: 1 if it overlaps nothing, 0.5 if it always shares its days with one other label. Use it as a sample weight.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def avg_uniqueness(t1, index):
    c = p.num_concurrent(t1, index)
    return pd.Series({t0: float((1 / c.loc[t0:end]).mean()) for t0, end in t1.items()})

t1 = tb["t1"]
mine = p.check("avg_uniqueness", p.attempt(avg_uniqueness, t1, close.index), p.avg_uniqueness(t1, close.index))
every_day = p.triple_barrier(close, close.index[100:-11], vol)
print(f"CUSUM events: mean uniqueness {mine.mean():.2f};  a label on every day: "
      f"{p.avg_uniqueness(every_day['t1'], close.index).mean():.2f} ({len(every_day)} heavily overlapping labels)")

Labeling every day makes each label mostly a copy of its neighbours; CUSUM events are much more independent. A standard bootstrap of overlapping labels draws redundant samples; the **sequential bootstrap** favours labels that add new days:

In [ ]:
sub = tb["t1"].iloc[:60]
idx = close.loc[sub.index[0]:sub.max()].index

def mean_uniqueness(draws):                           # uniqueness of the drawn sample, repeats included
    spans = [(sub.index[j], sub.iloc[j]) for j in draws]
    c = pd.Series(0, index=idx)
    for s, e in spans:
        c.loc[s:e] += 1
    return np.mean([(1 / c.loc[s:e]).mean() for s, e in spans])

seq = p.sequential_bootstrap(sub, idx, seed=0)
std = list(np.random.default_rng(0).integers(0, len(sub), len(sub)))
print(f"mean uniqueness of the drawn sample: sequential bootstrap {mean_uniqueness(seq):.2f}, standard bootstrap {mean_uniqueness(std):.2f}")

## 3. Meta-labeling

Keep a simple **primary** rule for the side (here 20-day momentum) and ask ML a narrower question: *given this signal, will the trade make money?* The meta-label is 1 when `side × barrier return > 0`. In the regime market, momentum should work in the trend regime and fail in chop. Does it?

In [ ]:
side = p.momentum_side(close).loc[tb.index]
meta = p.meta_labels(side, tb["ret"])
regime = mk["regime"].loc[tb.index].map({0: "chop (hidden)", 1: "trend (hidden)"})
meta.groupby(regime).agg(["mean", "size"]).rename(columns={"mean": "win rate", "size": "events"}).round(3)

Barely above 50% in trend and below it in chop: a meta-model that could tell the regimes apart would skip the chop trades. It does not have to be right often, only to size down when it is unsure. Turn a probability `p` into a size with `z = (p − 1/n)/√(p(1 − p))` (`n = 2` classes) and `size = clip(2Φ(z) − 1, 0, 1)` (`norm.cdf` from scipy).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from scipy.stats import norm

def bet_size(prob, n_classes=2):
    prob = np.asarray(prob, dtype=float)
    z = (prob - 1 / n_classes) / np.sqrt(prob * (1 - prob))
    return np.clip(2 * norm.cdf(z) - 1, 0, 1)

probs = np.array([0.3, 0.5, 0.55, 0.6, 0.7, 0.8, 0.9, 0.99])
mine = p.check("bet_size", p.attempt(bet_size, probs), p.bet_size(probs))
pd.Series(mine, index=probs, name="size").round(3).to_frame().T

Below 50% the size is zero (skip the trade); it grows slowly at first, so a barely-confident model trades small. Notebook 07 runs the full walk-forward meta-labeled strategy.

## Wrap-up

* Label with barriers scaled by volatility, and record when each label **ends** (`t1`): purging needs it.
* Overlapping labels are not independent: weight by uniqueness, bootstrap sequentially.
* Meta-labeling: a simple rule picks the side, ML decides whether and how much.
* Graded version: `labs/part10/week34_labels` (`triple_barrier`, `avg_uniqueness`, `sequential_bootstrap`, `meta_labels`, `bet_size`).